# Notebook 4 — Missing Value Imputation: From Simple to Sophisticated

**Difficulty:** ⭐⭐⭐ &nbsp;|&nbsp; **Estimated Time:** 3 hours &nbsp;|&nbsp; **Week 4 — Data Fundamentals & Preprocessing Pipelines**

---

## Why Does This Matter for ML?

Scikit-learn, XGBoost (by default), and most neural network layers **cannot handle NaN values**. They will either throw an error or produce wrong predictions. Before any model can run, you must fill in the gaps.

But the way you fill in those gaps matters enormously:

- **Mean imputation** fills a missing credit score with 650 — even if that row belonged to a 22-year-old student with no credit history and a $5 balance. The model will think this person looks average.
- **KNN imputation** finds 5 similar people and uses their scores — the model gets a much more realistic estimate.
- **Iterative imputation** builds a regression model specifically to predict credit score from all other features — the gold standard for complex datasets.

**Each strategy makes different trade-offs between accuracy, speed, and complexity.**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Apply mean, median, mode, and constant imputation with `SimpleImputer`
2. Apply KNN imputation and explain when it beats mean imputation
3. Apply iterative (MICE) imputation for complex feature relationships
4. Evaluate imputation quality using a hold-out mask test
5. Correctly fit imputers on training data only (avoiding data leakage)

## The Crossword Puzzle Analogy

Imputation is like filling in a crossword puzzle with missing letters.

| Crossword strategy | Imputation equivalent | When it works |
|---|---|---|
| Always write the most common letter ('E') regardless of context | **Mean/mode imputation** | When data is symmetric and missingness is low |
| Look at 5 nearby clues that are similar to this one | **KNN imputation** | When features have meaningful correlations |
| Go through all clues repeatedly, improving each answer with every pass | **Iterative (MICE) imputation** | When feature relationships are complex |

The first strategy is fast but dumb. The third is slow but often very accurate. Real-world practitioners start with the first, evaluate, then upgrade only if necessary.

## Section 1 — Setup: Synthetic Credit Card Dataset with Realistic Missingness

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# sklearn imputers — three strategies at increasing sophistication
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# IterativeImputer is experimental — must be enabled explicitly
from sklearn.experimental import enable_iterative_imputer  # noqa — required import

# Reproducibility
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('All imports successful.')
print('sklearn IterativeImputer enabled.')

In [ ]:
# ── Generate the base credit card transaction dataset ─────────────────────────
N = 3000  # slightly larger dataset for stable imputation evaluation

# --- Base features (no missingness yet) ---
age        = np.random.randint(18, 75, size=N)
income     = np.random.lognormal(mean=10.5, sigma=0.7, size=N).round(2)

# Transaction amount: log-normal — most are small, few are large
amount     = np.random.lognormal(mean=4.5, sigma=1.2, size=N).round(2)

# Credit score: correlated with income and age (older/richer → higher score)
# This correlation is intentional — it makes KNN and MICE beat mean imputation
credit_score_base = (
    450
    + 0.003 * income                   # higher income → better score
    + 1.2   * (age - 18)               # older → better score (more credit history)
    + np.random.normal(0, 40, N)       # add noise so it's not perfectly predictable
)
credit_score = np.clip(credit_score_base, 300, 850).round(0).astype(int)

# Account balance: correlated with income, log-normal, skewed right
balance_base = (
    np.exp(0.00005 * income + np.random.normal(5.5, 1.2, N))
)
account_balance = np.clip(balance_base, 1, 500_000).round(2)

# Merchant category: categorical feature with 5 categories
merchant_category = np.random.choice(
    ['grocery', 'travel', 'dining', 'online', 'gas_station'],
    size=N,
    p=[0.30, 0.15, 0.20, 0.25, 0.10]
)

# Fraud label: low base rate, higher for low balance + poor credit
fraud_prob = np.clip(
    0.03
    + 0.12 * (account_balance < 100)
    + 0.06 * (credit_score < 550),
    0, 1
)
is_fraud = np.random.binomial(1, fraud_prob)

# Keep the COMPLETE dataset — we will corrupt it below
df_complete = pd.DataFrame({
    'age': age,
    'income': income,
    'amount': amount,
    'credit_score': credit_score,
    'account_balance': account_balance,
    'merchant_category': merchant_category,
    'is_fraud': is_fraud
})

print(f'Complete dataset shape: {df_complete.shape}')
print(f'Fraud rate: {is_fraud.mean():.2%}')
print(f'\nCredit score: mean={credit_score.mean():.0f}, std={credit_score.std():.0f}')
print(f'Account balance: median=${np.median(account_balance):,.0f}, mean=${account_balance.mean():,.0f}')
df_complete.describe().round(2)

In [ ]:
# ── Introduce realistic missingness into the dataset ──────────────────────────
df = df_complete.copy()

# amount: MCAR, low missingness (~5%) — symmetric, suitable for mean imputation demo
amount_missing_idx = np.random.choice(df.index, size=int(0.05 * N), replace=False)
df.loc[amount_missing_idx, 'amount'] = np.nan

# credit_score: MAR — missing more often for younger users (age < 25)
# This creates systematic missingness that can be handled by model-based imputation
cs_missing_prob = np.where(df['age'] < 25, 0.35, 0.08)
cs_missing_mask = np.random.binomial(1, cs_missing_prob).astype(bool)
df.loc[cs_missing_mask, 'credit_score'] = np.nan

# account_balance: MNAR — low balance more likely missing (skewed, use median)
bal_norm = (df_complete['account_balance'] - df_complete['account_balance'].min()) / \
           (df_complete['account_balance'].max() - df_complete['account_balance'].min())
bal_missing_prob = 1 / (1 + np.exp(6 * bal_norm - 0.5))
bal_missing_mask = np.random.binomial(1, bal_missing_prob).astype(bool)
df.loc[bal_missing_mask, 'account_balance'] = np.nan

# merchant_category: MCAR, ~5% missing — categorical, use mode or 'UNKNOWN'
cat_missing_idx = np.random.choice(df.index, size=int(0.05 * N), replace=False)
df.loc[cat_missing_idx, 'merchant_category'] = np.nan

# Report final missingness
print('=== Missing Value Summary ===')
miss_summary = df.isnull().agg(['sum', 'mean']).T
miss_summary.columns = ['n_missing', 'pct_missing']
miss_summary = miss_summary[miss_summary['n_missing'] > 0]
miss_summary['pct_missing'] = miss_summary['pct_missing'].map('{:.1%}'.format)
miss_summary['mechanism'] = ['MCAR', 'MAR', 'MNAR', 'MCAR']
print(miss_summary.to_string())

---
## Section 2 — Strategy 1: Mean / Median / Mode Imputation

### Plain English Explanation

The simplest possible approach: replace every missing value with a single summary statistic computed from all the non-missing values.

| Strategy | When to use | Why |
|---|---|---|
| **Mean** | Symmetric, no severe outliers | Mean = center of a bell curve |
| **Median** | Skewed distributions, heavy outliers | Median is not pulled by extreme values |
| **Most Frequent (Mode)** | Categorical features | Only meaningful "center" for categories |
| **Constant** | Categorical with meaningful default | E.g., `'UNKNOWN'` preserves that it was missing |

### The Variance Reduction Problem

The main drawback: if 15% of `credit_score` values are all replaced with 650, the resulting column has **less spread** than the true data. This makes the model underestimate uncertainty and can reduce the predictive power of the feature.

In [ ]:
# ── SimpleImputer: mean/median/mode/constant ──────────────────────────────────
# sklearn's SimpleImputer is a transformer: fit() learns statistics, transform() applies them
# This fit/transform pattern is critical for preventing data leakage (covered in Section 6)

# --- Mean imputation for 'amount' (symmetric, MCAR) ---
mean_imputer = SimpleImputer(strategy='mean')
# fit_transform is equivalent to .fit(X).transform(X) — convenient for one-step use
# We reshape: sklearn expects 2D input, even for a single column
amount_mean_imputed = mean_imputer.fit_transform(df[['amount']]).flatten()

print(f'Amount — original mean: {df["amount"].mean():.2f}')
print(f'Amount — imputed with:  {mean_imputer.statistics_[0]:.2f}')
print(f'Amount — std (original non-missing): {df["amount"].std():.2f}')
print(f'Amount — std (after mean imputation): {pd.Series(amount_mean_imputed).std():.2f}')
print()

# --- Median imputation for 'account_balance' (skewed, MNAR) ---
median_imputer = SimpleImputer(strategy='median')
balance_median_imputed = median_imputer.fit_transform(df[['account_balance']]).flatten()

print(f'Balance — median used for imputation: ${median_imputer.statistics_[0]:,.2f}')
print(f'Balance — mean of true values:        ${df_complete["account_balance"].mean():,.2f}')
print(f'(Note: median << mean because distribution is right-skewed — median is more representative)')
print()

# --- Mode imputation for 'credit_score' ---
mode_imputer = SimpleImputer(strategy='most_frequent')
cs_mode_imputed = mode_imputer.fit_transform(df[['credit_score']]).flatten()
print(f'Credit score — mode used for imputation: {mode_imputer.statistics_[0]:.0f}')
print()

# --- Constant imputation for 'merchant_category' ---
# Using 'UNKNOWN' explicitly tells the model "this information was not captured"
# This can be more informative than using the most frequent category
constant_imputer = SimpleImputer(strategy='constant', fill_value='UNKNOWN')
merchant_const_imputed = constant_imputer.fit_transform(df[['merchant_category']]).flatten()
print(f'merchant_category — unique values after constant imputation:')
print(f'  {np.unique(merchant_const_imputed)}')

In [ ]:
# ── Visualize the variance-reduction problem of mean imputation ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mean Imputation: Variance Reduction Effect\n'
             'All missing values get the SAME number → artificial spike at the mean',
             fontsize=12, fontweight='bold')

# Left: histogram of TRUE amount vs mean-imputed amount (full distribution)
axes[0].hist(df_complete['amount'], bins=50, alpha=0.5, density=True,
             color='steelblue', label='True distribution (no missing)')
axes[0].hist(amount_mean_imputed, bins=50, alpha=0.5, density=True,
             color='tomato', label='After mean imputation')
axes[0].axvline(mean_imputer.statistics_[0], color='darkred', linewidth=2,
                linestyle='--', label=f'Imputed value (mean={mean_imputer.statistics_[0]:.0f})')
axes[0].set_title('Transaction Amount: True vs Mean-Imputed')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8.5)
axes[0].set_xlim(0, 2000)  # truncate extreme tail for readability

# Right: KDE of credit_score — true vs median-imputed
# The spike at the mode/median is visible in the imputed distribution
cs_not_missing_mask = ~df['credit_score'].isna()
sns.kdeplot(df_complete['credit_score'], ax=axes[1], label='True (complete)',
            color='steelblue', linewidth=2)
sns.kdeplot(pd.Series(cs_mode_imputed), ax=axes[1], label='After mode imputation',
            color='tomato', linewidth=2, linestyle='--')
axes[1].axvline(mode_imputer.statistics_[0], color='darkred', linewidth=2,
                linestyle=':', label=f'Mode = {mode_imputer.statistics_[0]:.0f}')
axes[1].set_title('Credit Score: True vs Mode-Imputed')
axes[1].set_xlabel('Credit Score')
axes[1].legend(fontsize=8.5)

plt.tight_layout()
plt.show()
print('Key observation: the imputed distribution has a bump/spike at the imputed value.')
print('This is the variance reduction problem — the distribution looks more peaked than reality.')

---
## Section 3 — Strategy 2: KNN Imputation

### Plain English First

Instead of using the **overall average** to fill in a missing value, KNN imputation finds the **K most similar rows** in your dataset and uses their values.

**The intuition:** if you know that a customer is:
- 28 years old
- Income \$45,000
- Account balance \$2,500

...then you can find 5 other customers with similar age, income, and balance, and use **their credit scores** to estimate the missing credit score. This is almost always more accurate than using the average across 3,000 customers.

### How It Works

1. For each row with a missing value, compute the **distance** to all other rows using the observed features
2. Select the K nearest neighbors (smallest distances)
3. Fill in the missing value with the **weighted average** of those neighbors' values
4. Neighbors closer in distance get more weight

### Drawback

Computing distances for every missing value requires scanning the entire dataset — **O(N × D)** per missing value, where D is the number of features. For 500,000 rows, this can take minutes.

In [ ]:
# ── KNN Imputation ────────────────────────────────────────────────────────────
# KNNImputer works ONLY on numeric features — encode or drop categoricals first
# Features used to compute distances: age, income, amount, account_balance
# The MISSING feature: credit_score (what we want to impute)

# Step 1: Select numeric columns for KNN (drop categorical and target)
knn_cols = ['age', 'income', 'amount', 'credit_score', 'account_balance']
df_knn_input = df[knn_cols].copy()

# Step 2: Create the KNN imputer
# n_neighbors=5: use 5 nearest rows to estimate missing value
# weights='distance': closer neighbors have more influence than distant ones
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

# Step 3: Fit and transform — sklearn will impute ALL columns with missing values
# (including amount and account_balance, not just credit_score)
df_knn_result = pd.DataFrame(
    knn_imputer.fit_transform(df_knn_input),
    columns=knn_cols
)

# Verify: no missing values remain
print(f'Missing values before KNN: {df_knn_input.isnull().sum().sum()}')
print(f'Missing values after KNN:  {df_knn_result.isnull().sum().sum()}')
print()

# Compare imputed credit_score distribution to true distribution
knn_imputed_cs = df_knn_result['credit_score']
mean_imputed_cs = mode_imputer.fit_transform(df[['credit_score']]).flatten()

print('=== Credit Score Imputation Comparison ===')
print(f'True (complete) — mean: {df_complete["credit_score"].mean():.1f},  std: {df_complete["credit_score"].std():.1f}')
print(f'Mode imputation — mean: {pd.Series(mean_imputed_cs).mean():.1f},  std: {pd.Series(mean_imputed_cs).std():.1f}')
print(f'KNN  imputation — mean: {knn_imputed_cs.mean():.1f},  std: {knn_imputed_cs.std():.1f}')
print()
print('KNN should have std CLOSER to the true std than mode imputation')
print('(because KNN creates varied imputed values, not a single repeated number)')

In [ ]:
# ── Visual comparison: KNN vs mean vs true distribution ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('KNN Imputation vs Mode Imputation for Credit Score',
             fontsize=13, fontweight='bold')

# Left: KDE overlay — true, mode-imputed, KNN-imputed
sns.kdeplot(df_complete['credit_score'], ax=axes[0],
            label='True distribution', color='steelblue', linewidth=2.5)
sns.kdeplot(pd.Series(mean_imputed_cs), ax=axes[0],
            label='Mode imputation', color='tomato', linewidth=2, linestyle='--')
sns.kdeplot(knn_imputed_cs, ax=axes[0],
            label='KNN imputation (k=5)', color='#2ECC71', linewidth=2, linestyle='-.')
axes[0].set_title('Distribution Preservation: True vs Imputed')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].text(0.03, 0.97,
             'KNN (green) tracks the true curve\nbetter than mode (red)',
             transform=axes[0].transAxes, va='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Right: scatter — true credit score vs KNN-imputed, for rows that were missing
# Using the FULL dataset allows us to check accuracy (we know the ground truth)
originally_missing_cs = df['credit_score'].isna()  # which rows were NaN?
true_vals_of_missing = df_complete.loc[originally_missing_cs, 'credit_score'].values
knn_vals_of_missing  = df_knn_result.loc[originally_missing_cs, 'credit_score'].values

axes[1].scatter(true_vals_of_missing, knn_vals_of_missing,
                alpha=0.3, s=20, color='#2ECC71', label='KNN imputed')
# Perfect predictions would lie on the diagonal y=x
lims = [300, 850]
axes[1].plot(lims, lims, 'k--', linewidth=1.5, label='Perfect prediction (y=x)')
axes[1].set_title('KNN Accuracy: True vs Imputed Value\n(for originally-missing rows)')
axes[1].set_xlabel('True credit_score')
axes[1].set_ylabel('KNN imputed credit_score')
axes[1].legend(fontsize=9)

mae_knn = mean_absolute_error(true_vals_of_missing, knn_vals_of_missing)
axes[1].text(0.03, 0.97, f'MAE = {mae_knn:.1f} points',
             transform=axes[1].transAxes, va='top', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.9))

plt.tight_layout()
plt.show()
print(f'KNN MAE on missing credit scores: {mae_knn:.1f} credit score points')

---
## Section 4 — Strategy 3: Iterative Imputation (MICE)

### Plain English First

**MICE = Multiple Imputation by Chained Equations.** It is the most sophisticated single-dataset imputation method.

**The process (in human terms):**

1. **Round 0:** Fill all missing values with the column mean (rough initialization)
2. **Round 1:** 
   - Take `credit_score` — temporarily set the imputed values back to NaN
   - Train a regression model: predict `credit_score` using `age`, `income`, `amount`, `balance`
   - Use that model to re-fill `credit_score`
   - Repeat for each other column with missingness
3. **Rounds 2–10:** Repeat the above — each round the estimates improve because all features now have better-quality values
4. **Convergence:** After ~10 iterations, the estimates stabilize

### Why It Works

The iterative approach exploits ALL relationships between features simultaneously. If credit score, balance, and income are all correlated, each round improves the other columns' estimates — which in turn improves the current column's model. It converges to a stable, high-quality solution.

### When to Use

- Complex datasets with many interrelated features
- MAR mechanism (missingness explained by other observed features)
- When accuracy matters more than computation time
- Statistical analysis where you need correct uncertainty estimates

In [ ]:
# ── Iterative Imputation (MICE) ───────────────────────────────────────────────
# sklearn's IterativeImputer implements a single-imputation version of MICE
# For full multiple imputation (with uncertainty), use the 'miceforest' library

# Use the same numeric columns as KNN
iter_cols = ['age', 'income', 'amount', 'credit_score', 'account_balance']
df_iter_input = df[iter_cols].copy()

# Create the iterative imputer
# max_iter=10: run 10 full cycles through all features
# random_state=42: reproducibility
# estimator=None uses sklearn's default BayesianRidge regression as the underlying model
iter_imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    tol=1e-3,          # stop early if estimates change less than this threshold
    imputation_order='roman'  # go through features left to right each pass
)

# fit_transform: learn all feature relationships, then impute
df_iter_result = pd.DataFrame(
    iter_imputer.fit_transform(df_iter_input),
    columns=iter_cols
)

print(f'Missing before iterative imputation: {df_iter_input.isnull().sum().sum()}')
print(f'Missing after  iterative imputation: {df_iter_result.isnull().sum().sum()}')
print()

# How many iterations did it actually run?
print(f'Iterations completed: {iter_imputer.n_iter_}')
print()

# Compare summary statistics across methods
iter_imputed_cs = df_iter_result['credit_score']
print('=== Credit Score Statistics Comparison ===')
print(f'{"Method":<25} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 62)
print(f'{"True (complete)":<25} {df_complete["credit_score"].mean():>8.1f} {df_complete["credit_score"].std():>8.1f} {df_complete["credit_score"].min():>8} {df_complete["credit_score"].max():>8}')
print(f'{"Mode imputation":<25} {pd.Series(mean_imputed_cs).mean():>8.1f} {pd.Series(mean_imputed_cs).std():>8.1f}')
print(f'{"KNN (k=5)":<25} {knn_imputed_cs.mean():>8.1f} {knn_imputed_cs.std():>8.1f}')
print(f'{"Iterative (MICE)":<25} {iter_imputed_cs.mean():>8.1f} {iter_imputed_cs.std():>8.1f}')

---
## Section 5 — Quantitative Comparison: Hold-Out Evaluation

### The Hold-Out Test for Imputation Quality

We cannot directly measure imputation accuracy in production (we don't know the true values). But we can simulate it:

1. Take the **complete data** (no NaNs)
2. Artificially mask 10% of known values (create fake missing values)
3. Apply each imputation strategy
4. Compare the imputed values to the **true values** using MAE
5. Lower MAE = better imputation

In [ ]:
# ── Hold-out evaluation: mask known values, impute, compute MAE ───────────────
# We evaluate on credit_score (most interesting: correlated with multiple features)

HOLDOUT_FRAC = 0.10  # mask 10% of the complete credit_score values

# Use only rows where credit_score is NOT already missing in df
# (we need the true value to evaluate)
eval_df = df_complete[['age', 'income', 'amount', 'credit_score', 'account_balance']].copy()

# Randomly select rows to mask (our artificial held-out test set)
holdout_size = int(HOLDOUT_FRAC * len(eval_df))
holdout_idx = np.random.choice(eval_df.index, size=holdout_size, replace=False)
true_cs_values = eval_df.loc[holdout_idx, 'credit_score'].values  # save ground truth

# Create the evaluation set with the held-out values masked
eval_df_masked = eval_df.copy()
eval_df_masked.loc[holdout_idx, 'credit_score'] = np.nan

print(f'Evaluation setup: {holdout_size} rows held out for testing')
print(f'Total missing in eval_df_masked: {eval_df_masked["credit_score"].isna().sum()}')
print()

mae_results = {}  # store MAE for each method

# --- Method 1: Mean imputation ---
si_mean = SimpleImputer(strategy='mean')
cs_mean_imputed = si_mean.fit_transform(eval_df_masked[['credit_score']]).flatten()
mae_results['Mean'] = mean_absolute_error(
    true_cs_values,
    cs_mean_imputed[holdout_idx]  # compare only on the masked rows
)

# --- Method 2: Median imputation ---
si_median = SimpleImputer(strategy='median')
cs_median_imputed = si_median.fit_transform(eval_df_masked[['credit_score']]).flatten()
mae_results['Median'] = mean_absolute_error(
    true_cs_values,
    cs_median_imputed[holdout_idx]
)

# --- Method 3: KNN imputation (uses all features) ---
knn_eval = KNNImputer(n_neighbors=5, weights='distance')
knn_eval_result = knn_eval.fit_transform(eval_df_masked)
mae_results['KNN (k=5)'] = mean_absolute_error(
    true_cs_values,
    knn_eval_result[holdout_idx, 3]  # column 3 = credit_score
)

# --- Method 4: Iterative imputation ---
iter_eval = IterativeImputer(max_iter=10, random_state=42)
iter_eval_result = iter_eval.fit_transform(eval_df_masked)
mae_results['Iterative\n(MICE)'] = mean_absolute_error(
    true_cs_values,
    iter_eval_result[holdout_idx, 3]  # column 3 = credit_score
)

print('=== Imputation MAE Results (credit_score) ===')
print('Lower MAE = imputed values closer to true values')
print()
for method, mae in mae_results.items():
    method_clean = method.replace('\n', ' ')
    print(f'  {method_clean:<20}: MAE = {mae:.2f} credit score points')

In [ ]:
# ── Visualization: MAE comparison bar chart + distribution recovery ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Imputation Strategy Comparison: Accuracy & Distribution Recovery',
             fontsize=12, fontweight='bold')

# Left: MAE bar chart
methods = list(mae_results.keys())
maes    = list(mae_results.values())
colors  = ['#E74C3C', '#E67E22', '#2ECC71', '#3498DB']  # red → green = worse → better

bars = axes[0].bar(methods, maes, color=colors, edgecolor='black', linewidth=0.7, width=0.55)
for bar, mae in zip(bars, maes):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{mae:.1f}',
        ha='center', va='bottom', fontweight='bold', fontsize=11
    )
axes[0].set_title('Mean Absolute Error by Imputation Method\n(lower is better)')
axes[0].set_ylabel('MAE (credit score points)')
axes[0].set_ylim(0, max(maes) * 1.20)
# Add a "random guess" baseline for reference
random_baseline = df_complete['credit_score'].std() * np.sqrt(np.pi / 2)  # MAE of mean-0 noise

# Right: KDE of recovered distribution — how well does each method reproduce the true shape?
sns.kdeplot(eval_df['credit_score'], ax=axes[1],
            label='True (complete)', color='black', linewidth=2.5, linestyle='-')
sns.kdeplot(pd.Series(cs_mean_imputed), ax=axes[1],
            label=f'Mean (MAE={mae_results["Mean"]:.0f})', color='#E74C3C',
            linewidth=1.8, linestyle='--')
sns.kdeplot(pd.Series(knn_eval_result[:, 3]), ax=axes[1],
            label=f'KNN (MAE={mae_results["KNN (k=5)"]:.0f})', color='#2ECC71',
            linewidth=1.8, linestyle='-.')
sns.kdeplot(pd.Series(iter_eval_result[:, 3]), ax=axes[1],
            label=f'Iterative (MAE={mae_results["Iterative\\n(MICE)"]:.0f})', color='#3498DB',
            linewidth=1.8, linestyle=':')
axes[1].set_title('Credit Score Distribution Recovery')
axes[1].set_xlabel('Credit Score')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Takeaway: KNN and Iterative tend to better preserve distribution shape')
print('and achieve lower MAE, especially when features are correlated.')

---
## Section 6 — CRITICAL: Fit on Train Only — Avoiding Data Leakage

### The Golden Rule

> **ALWAYS fit your imputer on the training set only. NEVER fit on the full dataset before splitting.**

### Why This Matters (The Leak)

Suppose your test set happens to contain some unusually high credit scores (e.g., 820–850). When you fit the `SimpleImputer` on the **full dataset**, the computed mean includes those high values. The imputed mean is therefore slightly higher than it would be if computed from the training set alone.

Now your model has seen information from the test set during training — **even though no individual test row was used for training**. The model is tuned to a distribution that includes test data. When you evaluate on the test set, you get an **overly optimistic performance estimate**.

This is a subtle form of **data leakage** — the most common mistake in data science pipelines.

In [ ]:
# ── WRONG WAY: fit imputer on full dataset, then split ────────────────────────
# This is what many beginners (and sometimes experienced practitioners) do

print('=== WRONG APPROACH: Fitting imputer BEFORE train/test split ===')
print()

# Create a simple feature matrix and target for demonstration
feature_cols = ['age', 'income', 'amount', 'credit_score', 'account_balance']
X = df[feature_cols].copy()
y = df['is_fraud'].copy()

# WRONG: fit the imputer on ALL of X (train + test combined)
wrong_imputer = SimpleImputer(strategy='mean')
X_imputed_wrong = wrong_imputer.fit_transform(X)  # <<< uses test data statistics!

# Split AFTER imputation — the test data has already contaminated the imputer
X_train_wrong, X_test_wrong, y_train, y_test = train_test_split(
    X_imputed_wrong, y, test_size=0.20, random_state=42
)

# Report the imputer statistics (computed from full dataset)
print('WRONG imputer — statistics learned from FULL dataset (train + test):')
for col, stat in zip(feature_cols, wrong_imputer.statistics_):
    print(f'  {col:<20}: {stat:.3f}')

print()
print('Problem: the mean for credit_score includes test rows.')
print('         The model is tuned to test-set statistics before evaluation.')

In [ ]:
# ── RIGHT WAY: split first, fit imputer on train only ────────────────────────
print('=== CORRECT APPROACH: Fitting imputer AFTER train/test split ===')
print()

# Step 1: Split the RAW data (with NaNs still present)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Step 2: Create a FRESH imputer and fit it ONLY on the training set
# The imputer learns: "what is the mean credit_score in training data?"
right_imputer = SimpleImputer(strategy='mean')
right_imputer.fit(X_train_raw)  # <<< only sees training rows

# Step 3: Transform BOTH train and test using the SAME training statistics
# .transform() uses the statistics learned during .fit() — does NOT re-learn
X_train_imputed = right_imputer.transform(X_train_raw)
X_test_imputed  = right_imputer.transform(X_test_raw)   # uses TRAIN statistics only!

# Report the imputer statistics (computed from training data only)
print('CORRECT imputer — statistics learned from TRAIN set only:')
for col, stat in zip(feature_cols, right_imputer.statistics_):
    print(f'  {col:<20}: {stat:.3f}')

print()
print('The test set never influences the imputer statistics.')
print('Missing test values are filled with TRAIN-set statistics — realistic production behavior.')
print()

# Show the difference between wrong and right statistics
print('=== Comparison: Wrong vs Correct Imputer Statistics ===')
print(f'{"Feature":<22} {"Wrong (full)":>14} {"Correct (train)":>16} {"Diff":>8}')
print('-' * 65)
for col, wrong_s, right_s in zip(feature_cols, wrong_imputer.statistics_, right_imputer.statistics_):
    diff = wrong_s - right_s
    print(f'{col:<22} {wrong_s:>14.3f} {right_s:>16.3f} {diff:>+8.3f}')
print()
print('Even small differences accumulate when millions of rows are imputed.')

In [ ]:
# ── Visualize the leakage effect ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Data Leakage Visualization: Wrong vs Correct Imputation Order',
             fontsize=12, fontweight='bold')

# Extract credit_score column index for comparison
cs_col_idx = feature_cols.index('credit_score')

# Left: compare imputed values in the test set
# Wrong method uses full-dataset mean; right method uses train-only mean
wrong_test_cs  = X_test_wrong[:, cs_col_idx]
right_test_cs  = X_test_imputed[:, cs_col_idx]

axes[0].hist(wrong_test_cs,  bins=50, alpha=0.6, density=True,
             color='tomato', label='Wrong (full-data imputer)')
axes[0].hist(right_test_cs,  bins=50, alpha=0.6, density=True,
             color='steelblue', label='Correct (train-only imputer)')
axes[0].axvline(wrong_imputer.statistics_[cs_col_idx], color='darkred',
                linewidth=2, linestyle='--', label=f'Wrong mean: {wrong_imputer.statistics_[cs_col_idx]:.1f}')
axes[0].axvline(right_imputer.statistics_[cs_col_idx], color='darkblue',
                linewidth=2, linestyle=':', label=f'Train mean: {right_imputer.statistics_[cs_col_idx]:.1f}')
axes[0].set_title('Test Set credit_score Distribution\nWrong vs Correct Imputer')
axes[0].set_xlabel('Credit Score')
axes[0].legend(fontsize=8)
axes[0].set_xlim(300, 850)

# Right: flowchart-style diagram showing correct vs wrong pipeline
axes[1].axis('off')
wrong_text = (
    "❌  WRONG PIPELINE\n"
    "─────────────────────────────\n"
    "Raw data (train + test)\n"
    "         ↓\n"
    "  imputer.fit_transform(X_all)  ← LEAKS\n"
    "         ↓\n"
    "   train_test_split(X_imputed)\n"
    "         ↓\n"
    "  fit model on train\n"
    "         ↓\n"
    "  evaluate on test  ← BIASED\n"
)
right_text = (
    "✅  CORRECT PIPELINE\n"
    "─────────────────────────────\n"
    "Raw data\n"
    "    ↓\n"
    "train_test_split(X_raw)\n"
    "    ↓              ↓\n"
    "X_train (NaN)   X_test (NaN)\n"
    "    ↓              ↓\n"
    "imputer.fit(X_train)          \n"
    "    ↓              ↓\n"
    "transform(X_train) transform(X_test)\n"
    "    ↓              ↓\n"
    " fit model      evaluate  ← CLEAN\n"
)
axes[1].text(0.02, 0.98, wrong_text, transform=axes[1].transAxes, va='top',
             fontsize=8, fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.9))
axes[1].text(0.52, 0.98, right_text, transform=axes[1].transAxes, va='top',
             fontsize=8, fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.9))
axes[1].set_title('Pipeline Order Matters')

plt.tight_layout()
plt.show()

---
## Section 7 — Strategy Decision Table

| Situation | Recommended Strategy | Why |
|---|---|---|
| Numeric, symmetric, <10% missing, MCAR | `SimpleImputer(strategy='mean')` | Simple, fast, minimal bias |
| Numeric, skewed distribution, MNAR/MAR | `SimpleImputer(strategy='median')` | Robust to outliers |
| Categorical, no meaningful default | `SimpleImputer(strategy='most_frequent')` | Only sensible "center" for categories |
| Categorical, missingness is informative | `SimpleImputer(strategy='constant', fill_value='UNKNOWN')` | Preserves that value was missing |
| Features are correlated, MAR, <100K rows | `KNNImputer(n_neighbors=5)` | Uses feature relationships |
| Complex dataset, many correlated features | `IterativeImputer(max_iter=10)` | Best accuracy, highest compute cost |
| MNAR mechanism | Add `_was_missing` indicator FIRST, then any of the above | The missingness itself is predictive |
| Production pipeline | Wrap in `sklearn.pipeline.Pipeline` | Guarantees fit-on-train-only automatically |

---
## Section 8 — Putting It All Together: Final Summary

In [ ]:
# ── Comprehensive MAE comparison chart ───────────────────────────────────────
# Rebuild with consistent holdout for a clean final comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Final Imputation Strategy Comparison — Credit Score Hold-Out Test',
             fontsize=12, fontweight='bold')

methods_clean = ['Mean', 'Median', 'KNN (k=5)', 'Iterative\n(MICE)']
maes_clean    = [mae_results[m] for m in mae_results]
colors_bar    = ['#E74C3C', '#E67E22', '#27AE60', '#2980B9']

# Left: ranked bar chart (sorted best to worst)
sorted_pairs  = sorted(zip(maes_clean, methods_clean, colors_bar))
maes_s, meths_s, cols_s = zip(*sorted_pairs)

bars = axes[0].barh(list(meths_s), list(maes_s), color=list(cols_s),
                    edgecolor='black', linewidth=0.6, height=0.55)
for bar, mae in zip(bars, maes_s):
    axes[0].text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f'{mae:.1f} pts',
        va='center', ha='left', fontweight='bold'
    )
axes[0].set_title('MAE on Held-Out Credit Scores (lower = better)')
axes[0].set_xlabel('Mean Absolute Error (credit score points)')
axes[0].set_xlim(0, max(maes_s) * 1.25)
axes[0].invert_yaxis()  # best (lowest MAE) at top

# Right: speed vs accuracy trade-off bubble chart
# Rough relative timing (1 = fastest reference)
relative_speed = [1, 1, 25, 60]   # Mean and Median are ~equal, KNN and MICE are slower
maes_order     = [mae_results['Mean'], mae_results['Median'],
                  mae_results['KNN (k=5)'], mae_results['Iterative\n(MICE)']]
scatter_colors = ['#E74C3C', '#E67E22', '#27AE60', '#2980B9']
scatter_labels = ['Mean', 'Median', 'KNN', 'MICE']

for x, y, c, lbl in zip(relative_speed, maes_order, scatter_colors, scatter_labels):
    axes[1].scatter(x, y, color=c, s=300, zorder=5, edgecolors='black', linewidths=0.8)
    axes[1].annotate(lbl, (x, y), textcoords='offset points', xytext=(8, 5), fontsize=10)

axes[1].set_title('Speed vs Accuracy Trade-off\n(ideal: bottom-left corner)')
axes[1].set_xlabel('Relative Computation Time (1 = Mean imputer)')
axes[1].set_ylabel('MAE (lower = more accurate)')
# Ideal quadrant annotation
axes[1].annotate('← Better accuracy, faster',
                 xy=(0.05, 0.05), xycoords='axes fraction',
                 fontsize=8, color='darkgreen', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# ── Final summary printout ────────────────────────────────────────────────────
print('=' * 65)
print('NOTEBOOK 4 SUMMARY: Missing Value Imputation')
print('=' * 65)
print()
print('Strategy             | MAE  | Speed  | Complexity | Use when')
print('-' * 65)
print('Mean imputation      | High |  Fast  |   Low      | Symmetric, MCAR, quick baseline')
print('Median imputation    | High |  Fast  |   Low      | Skewed, outliers present')
print('KNN imputation       | Low  | Medium |   Medium   | Correlated features, MAR')
print('Iterative/MICE       | Low  |  Slow  |   High     | Complex feature relationships')
print()
print('GOLDEN RULE: Always fit imputers on TRAIN data only.')
print('             Fitting on full data → data leakage → over-optimistic evaluation.')
print()
print('For MNAR: add _was_missing indicator column BEFORE any imputation.')
print('For production: wrap imputers in sklearn Pipeline to enforce correct order.')
print()
print('Hold-out MAE results:')
for method, mae in mae_results.items():
    method_clean = method.replace('\n', ' ')
    bar = '█' * int(mae / 3)  # simple ASCII bar
    print(f'  {method_clean:<22}: {mae:5.1f}  {bar}')
print()
print('Next: Feature Engineering — creating new features from existing ones.')

---
## Self-Check Questions

Test your understanding. Try to answer without looking at the notebook.

---

**Question 1:** `age` in your credit card dataset has 20% missing values and is roughly symmetric (bell-shaped). Which imputation strategy would you start with and why? What would you check before deciding?

<details>
<summary>Click to reveal answer</summary>

**Start with `SimpleImputer(strategy='mean')`.**

**Why mean:** age is symmetric (the description says bell-shaped), which means mean ≈ median ≈ mode. The mean is the minimum-variance unbiased estimator for the center of a symmetric distribution.

**But before finalizing, check:**
1. What is the **missingness mechanism**? If MNAR (e.g., elderly users are less likely to provide age), you need to add a binary `age_was_missing` indicator first, and consider whether imputing with mean introduces bias.
2. How correlated is `age` with other features? If it is highly correlated (e.g., with `credit_score`), KNN or iterative imputation would give more accurate imputed values.
3. Does 20% missing violate your model's tolerance? With high missingness, consider whether the model would benefit from KNN's variance-preserving imputation.

**Answer:** Mean imputation is the right starting point. Upgrade to KNN or iterative only if initial model performance is unsatisfactory.
</details>

---

**Question 2:** KNN imputation is more accurate than mean imputation, but your dataset has 500,000 rows. What is the practical concern, and what are your options?

<details>
<summary>Click to reveal answer</summary>

**Practical concern: computational cost.**

`KNNImputer` computes pairwise distances between every row with a missing value and all other rows in the dataset. For 500,000 rows and 20 features, this is an O(N × D) operation per missing row — which can take minutes to hours in production.

**Options:**
1. **Use mean/median imputation** if accuracy difference is small (check with hold-out test)
2. **Subsample for fitting:** fit KNN on a representative sample (e.g., 50,000 rows), then use it to transform all 500,000
3. **Use IterativeImputer with a fast estimator** — replace the default BayesianRidge with a `ExtraTreesRegressor(n_estimators=10)` which can be faster on large datasets
4. **Use `miceforest`** which implements MICE with LightGBM — much faster on large datasets
5. **Engineer the missing feature away:** if `credit_score` can be approximated from `age + income` with an explicit formula, use that formula instead of imputation

Practical rule: always profile imputation time on a 10% sample before committing to KNN on large datasets.
</details>

---

**Question 3:** You fit a `KNNImputer` on your full dataset (train + test combined), then split into train and test sets. Why is this a problem, even though no individual test row was used to train your downstream model?

<details>
<summary>Click to reveal answer</summary>

**This is data leakage — specifically, preprocessing leakage.**

When `KNNImputer` finds the K nearest neighbors for a training-set row with a missing value, it is allowed to use **test-set rows as neighbors**. This means the imputed values in the training set are influenced by test data.

**Consequences:**
1. The imputed values in the training set reflect test-set patterns → the model trains on a slightly "test-contaminated" dataset
2. The model's evaluation on the test set is optimistic — it has already "seen" test patterns via the imputer
3. In production (where test data doesn't exist yet), the imputer will behave differently → production performance is worse than evaluated performance

**Correct pipeline:**
```python
X_train, X_test = train_test_split(X_raw, ...)
imputer = KNNImputer()
imputer.fit(X_train)           # learns from train only
X_train = imputer.transform(X_train)
X_test  = imputer.transform(X_test)  # uses train-learned neighbors
```
</details>

---

**Question 4:** For `merchant_category` (categorical column with 5 categories, 5% missing values, MCAR mechanism), which `SimpleImputer` strategy would you choose and why? Would you consider using `fill_value='UNKNOWN'` instead?

<details>
<summary>Click to reveal answer</summary>

**For MCAR at 5% missing: `strategy='most_frequent'` is the standard choice.**

**Why most_frequent:**
- MCAR means the missing values are a random sample of all merchant categories
- The most frequent category (e.g., 'grocery' at 30%) is the maximum likelihood estimate for any randomly selected category
- 5% missing is low enough that filling all with the mode has negligible impact on model training

**When `fill_value='UNKNOWN'` is better:**
- If the missingness is **MAR or MNAR** (there is a reason this category is missing) — then 'UNKNOWN' preserves the signal that it was missing
- If downstream you want to track data quality (missing vs imputed)
- If you are concerned that 'grocery' appearing more often than it should will bias a category-based model

**Recommendation:** use `most_frequent` for this case (MCAR, 5% missing). Use `'UNKNOWN'` when the missing category could itself be a signal — e.g., if missing merchant category correlates with higher fraud rates.
</details>